# Repeated `idcontrato` in the raw workbooks

`bulk-ingest` keeps one row per `idcontrato` (last wins), so `ingested < read`. This notebook looks
only at the repeated ids in `data/raw/contracts_raw_<YEAR>.parquet`: how many, what differs
between the copies, and what is lost by keeping one.

In [1]:
from pathlib import Path

import polars as pl

from base_gov import raw_ingest

DATA = Path("..") / "data"
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_fmt_str_lengths(80)

years = raw_ingest.raw_years(DATA / "raw")
raw = pl.concat(
    [pl.read_parquet(raw_ingest.raw_path(DATA / "raw", y)).with_columns(src_year=pl.lit(y)) for y in years],
    how="diagonal_relaxed",
)
dups = raw.filter(pl.col("idcontrato").is_duplicated()).sort("idcontrato")
print(f"years={years}  raw rows={raw.height:,}  repeated rows={dups.height:,}  ids={dups['idcontrato'].n_unique():,}")

years=[2025, 2026]  raw rows=410,244  repeated rows=1,800  ids=900


## 1. How many, and how many copies per id

In [2]:
per_year = raw.group_by("src_year").agg(
    pl.len().alias("rows"),
    pl.col("idcontrato").is_duplicated().sum().alias("rows_in_repeated_ids"),
    pl.col("idcontrato").filter(pl.col("idcontrato").is_duplicated()).n_unique().alias("repeated_ids"),
).sort("src_year")
display(per_year)
display(dups.group_by("idcontrato").len()["len"].value_counts().sort("len"))
# Do any ids repeat ACROSS years (i.e. would `keep=last` pick by file order)?
cross = dups.group_by("idcontrato").agg(pl.col("src_year").n_unique().alias("n_years")).filter(pl.col("n_years") > 1)
print("ids spanning more than one year file:", cross.height)

src_year,rows,rows_in_repeated_ids,repeated_ids
i32,u32,u32,u32
2025,247205,682,341
2026,163039,1118,559


len,count
u32,u32
2,900


ids spanning more than one year file: 0


## 2. Which columns differ between the copies

In [3]:
value_cols = [c for c in raw.columns if c not in ("idcontrato", "src_year")]
differing = dups.group_by("idcontrato").agg([pl.col(c).n_unique().alias(c) for c in value_cols])
diff_counts = (
    differing.select([(pl.col(c) > 1).sum().alias(c) for c in value_cols])
    .transpose(include_header=True, column_names=["ids_where_it_differs"])
    .filter(pl.col("ids_where_it_differs") > 0)
    .sort("ids_where_it_differs", descending=True)
)
diff_counts

column,ids_where_it_differs
str,u32
"""TipoCriterioAdjudicacao""",583
"""ContratEcologico""",317


## 3. Side-by-side examples\n\nFirst cases where only `TipoCriterioAdjudicacao` differs.

In [4]:
only_criterio = differing.filter(
    pl.all_horizontal([(pl.col(c) == 1) for c in value_cols if c != "TipoCriterioAdjudicacao"])
    & (pl.col("TipoCriterioAdjudicacao") > 1)
)["idcontrato"]
print("ids that differ ONLY in TipoCriterioAdjudicacao:", only_criterio.len())
cols = ["idcontrato", "src_year", "objectoContrato", "adjudicante", "precoContratual", "tipoprocedimento", "TipoCriterioAdjudicacao"]
for cid in only_criterio.head(5):
    display(dups.filter(pl.col("idcontrato") == cid).select(cols))

ids that differ ONLY in TipoCriterioAdjudicacao: 583


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
11142456,2025,"""CPR 012/2024 - Aquisição de vinhos, cervejas e bebidas brancas""","""680047360 - Serviços de Ação Social da Universidade do Minho""",11186.94,"""Consulta Prévia""","""Multifator"""
11142456,2025,"""CPR 012/2024 - Aquisição de vinhos, cervejas e bebidas brancas""","""680047360 - Serviços de Ação Social da Universidade do Minho""",11186.94,"""Consulta Prévia""","""Monofator"""


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
11149318,2025,"""3024011464/Fornecimento de Condecorações Militares""","""600012662 - MARINHA - Direção de Navios""",31228.0,"""Concurso público""","""Monofator"""
11149318,2025,"""3024011464/Fornecimento de Condecorações Militares""","""600012662 - MARINHA - Direção de Navios""",31228.0,"""Concurso público""","""Multifator"""


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
11149383,2025,"""3024011464/Fornecimento de Condecorações Militares""","""600012662 - MARINHA - Direção de Navios""",32075.0,"""Concurso público""","""Monofator"""
11149383,2025,"""3024011464/Fornecimento de Condecorações Militares""","""600012662 - MARINHA - Direção de Navios""",32075.0,"""Concurso público""","""Multifator"""


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
11171576,2025,"""24DC41CPI001-Prestação de Serviços de Manutenção e Assistência Técnica a Equipam…","""500745471 - Santa Casa da Misericórdia de Lisboa""",89400.0,"""Concurso público""","""Monofator"""
11171576,2025,"""24DC41CPI001-Prestação de Serviços de Manutenção e Assistência Técnica a Equipam…","""500745471 - Santa Casa da Misericórdia de Lisboa""",89400.0,"""Concurso público""","""Multifator"""


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
11178981,2025,"""24DC41CPI001-Prestação de Serviços de Manutenção e Assistência Técnica a Equipam…","""500745471 - Santa Casa da Misericórdia de Lisboa""",34974.0,"""Concurso público""","""Monofator"""
11178981,2025,"""24DC41CPI001-Prestação de Serviços de Manutenção e Assistência Técnica a Equipam…","""500745471 - Santa Casa da Misericórdia de Lisboa""",34974.0,"""Concurso público""","""Multifator"""


## 4. The `TipoCriterioAdjudicacao` pairs\n\nIf each copy carries one award criterion, the pair is a multi-criteria contract split over rows.

In [5]:
pairs = (
    dups.sort("idcontrato", "TipoCriterioAdjudicacao", nulls_last=True)
    .group_by("idcontrato", maintain_order=True)
    .agg(pl.col("TipoCriterioAdjudicacao").alias("criterios"))
    .with_columns(pair=pl.col("criterios").list.join("  |  "))
)
pairs["pair"].value_counts(sort=True).head(15)

pair,count
str,u32
"""Monofator | Multifator""",583
"""Multifator | Multifator""",172
"""Monofator | Monofator""",145


## 5. Other differing columns (`ContratEcologico`, ...)

In [6]:
other = [c for c in diff_counts["column"] if c != "TipoCriterioAdjudicacao"]
print("other differing columns:", other)
if other:
    ids = differing.filter(pl.any_horizontal([pl.col(c) > 1 for c in other]))["idcontrato"]
    display(dups.filter(pl.col("idcontrato").is_in(ids.head(5).implode())).select("idcontrato", "src_year", *other, "precoContratual"))

other differing columns: ['ContratEcologico']


idcontrato,src_year,ContratEcologico,precoContratual
i64,i32,str,f64
11158565,2025,"""Sim""",24000.0
11158565,2025,"""Não""",24000.0
11548289,2025,"""Sim""",2498670.4
11548289,2025,"""Não""",2498670.4
12225059,2025,"""Não""",21060.0
12225059,2025,"""Sim""",21060.0
12225065,2025,"""Sim""",20871.2
12225065,2025,"""Não""",20871.2
13843478,2026,"""Não""",30792.64


## 6. Impact\n\nWhat keeping *all* copies would do to counts and value, versus `keep=last`.

In [7]:
kept = raw.unique(subset="idcontrato", keep="last", maintain_order=True)
dropped = raw.height - kept.height
double_counted = dups.height - dups["idcontrato"].n_unique()
value_all = dups["precoContratual"].sum()
value_kept = kept.filter(pl.col("idcontrato").is_in(dups["idcontrato"].unique().implode()))["precoContratual"].sum()
print(f"rows dropped by keep=last: {dropped:,} (== extra copies: {double_counted:,})")
print(f"value on repeated ids: all copies €{value_all:,.0f} vs one copy €{value_kept:,.0f} "
      f"(double count €{value_all - value_kept:,.0f})")

rows dropped by keep=last: 900 (== extra copies: 900)
value on repeated ids: all copies €463,504,742 vs one copy €231,752,371 (double count €231,752,371)


## 7. Reading

_To fill in after review:_ is the extra row a multi-criteria split (lossless to collapse if the
criteria are merged into one cell) or a genuine duplicate? If the former, prefer aggregating
`TipoCriterioAdjudicacao` per `idcontrato` over `keep="last"`, which today discards one criterion
arbitrarily.